In [1]:
from mdagent import MDAgent

In [2]:
prompt3 = "Download the PDB file for protein 1GZX. Then, analyze the secondary structure of the protein and provide information on how many helices, sheets, and other components are present."
llm_var = "claude-3-opus-20240229"
tools = "all"
agent = MDAgent(agent_type="Structured", model=llm_var, top_k_tools=tools, use_memory=False)
print (llm_var)

claude-3-opus-20240229


In [3]:
agent.run(prompt3)

Thought: To answer this question, I will first need to download the PDB file for protein 1GZX. Then I can analyze the secondary structure of the protein to determine the number of helices, sheets, and other components.

Action:
```
{
  "action": "PDBFileDownloader",
  "action_input": "1GZX"
}
```

PDB file found with this ID: 1GZX
Thought: Now that I have downloaded the PDB file for 1GZX, I can analyze the secondary structure to determine the number of helices, beta sheets, and turns in the protein.

Action:
```
{
  "action": "GetTurnsBetaSheetsHelices",
  "action_input": {
    "query": "1GZX", 
    "primary_accession": "P00720"
  }
}
```

The primary accession number provided does not match any entry in the data, using the first entry instead.
The primary accession number provided does not match any entry in the data, using the first entry instead.
The primary accession number provided does not match any entry in the data, using the first entry instead.
Based on the information gather

({'input': '\n    You are an expert molecular dynamics scientist, and\n    your task is to respond to the question or\n    solve the problem to the best of your ability using\n    the provided tools.\n\n    You can only respond with a single complete\n    \'Thought, Action, Action Input\' format\n    OR a single \'Final Answer\' format.\n\n    Complete format:\n    Thought: (reflect on your progress and decide what to do next)\n    Action:\n    ```\n    {\n        "action": (the action name, it should be the name of a tool),\n        "action_input": (the input string for the action)\n    }\n    \'\'\'\n\n    OR\n\n    Final Answer: (the final response to the original input\n    question, once all steps are complete)\n\n    You are required to use the tools provided,\n    using the most specific tool\n    available for each action.\n    Your final answer should contain all information\n    necessary to answer the question and its subquestions.\n    Before you finish, reflect on your pro

In [4]:
registry = agent.path_registry
all_paths = registry.list_path_names_and_descriptions()
print (all_paths)
assert "1GZX" in all_paths
file_id = all_paths.split("Files found in registry: ")[1].split(":")[0]

Files found in registry: 1GZX_230625: PDB file downloaded from RSCB, PDBFile ID: 1GZX_230625


In [5]:
import mdtraj as md
file_path = registry.get_mapped_path(file_id)
traj = md.load(file_path)
top = traj.topology

secondary_structure = md.compute_dssp(traj,simplified=True)
print("Number of residues in sheets: ",len([i for i in secondary_structure[0] if i == 'E']))
print("Number of residues in helices: ",len([i for i in secondary_structure[0] if i == 'H']))
print("Number of residues in coils: ",len([i for i in secondary_structure[0] if i == 'C']))

Number of residues in sheets:  0
Number of residues in helices:  444
Number of residues in coils:  130


In [7]:
from mdagent.tools.base_tools.preprocess_tools.uniprot import GetTurnsBetaSheetsHelices

get_ss = GetTurnsBetaSheetsHelices()
all_info = get_ss._run("1GZX")
n_helices = all_info.split("Helices: ")[1].count("evidences")
n_helices

11